# yt-clips Kaggle Worker (T4-Optimized)
## Settings → Accelerator → GPU × 2 (T4)
Run all cells. Worker listens for jobs from your Mac.
Add host reference photos to the `photos/` folder for automatic face detection & tracking.

In [ ]:
# CELL 1: Check GPU + Clone Repo
import subprocess, os, sys, shutil
from pathlib import Path

# Check GPU
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null",
                     shell=True, capture_output=True, text=True).stdout.strip()
print(f"GPU: {gpu or 'NONE! Enable GPU × 2 in Settings → Accelerator'}")

# Set working dir
os.chdir("/kaggle/working")
print(f"Working dir: {os.getcwd()}")

# Clone repo if not present
if not Path("pipeline.py").exists():
    print("Cloning yt-clips repo...")
    subprocess.run("git clone --depth 1 https://github.com/fearless16/yt-clips.git _repo", shell=True)
    for item in Path("_repo").iterdir():
        dest = Path(item.name)
        if dest.exists():
            shutil.rmtree(dest) if dest.is_dir() else dest.unlink()
        shutil.move(str(item), ".")
    shutil.rmtree("_repo")

print(f"CWD: {os.getcwd()}")

In [ ]:
# CELL 2: Install System Dependencies
def run(cmd, desc=""):
    print(f"  -> {desc}...") if desc else None
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0 and r.stderr:
        print(f"    ⚠ stderr: {r.stderr.strip()[:200]}")
    return r

print("Installing system deps...")
run("apt-get update -qq > /dev/null 2>&1", "apt update")
run("apt-get install -y -qq aria2 ffmpeg > /dev/null 2>&1", "aria2 + ffmpeg")
run("npm install -g localtunnel > /dev/null 2>&1", "localtunnel")
print("  ✅ System deps installed")

In [ ]:
# CELL 3: Install Python Dependencies (T4-Optimized)
# Removed: RIFE (too slow on T4), CodeFormer (redundant with GFPGAN),
#          DeepFace (heavy TF), Video2X (unavailable)
def run(cmd, desc=""):
    print(f"  -> {desc}...") if desc else None
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0 and r.stderr:
        print(f"    ⚠ stderr: {r.stderr.strip()[:200]}")
    return r

print("Installing T4-optimized deps...")

# Core deps (no torch reinstall — Kaggle has it)
run("pip install -q yt-dlp faster-whisper rich PyYAML opencv-python-headless numpy "
    "filterpy scipy google-genai google-generativeai openai python-dotenv "
    "ultralytics face_recognition",
    "Core deps")

# basicsr (builds Cython for realesrgan/gfpgan)
r = run("pip install -q basicsr", "basicsr")
if r.returncode != 0:
    print("    Retrying with --no-build-isolation...")
    run("pip install -q --no-build-isolation basicsr", "basicsr retry")

# Real-ESRGAN + GFPGAN + FaceXLib
run("pip install -q realesrgan gfpgan facexlib",
    "RealESRGAN + GFPGAN + FaceXLib")

print("  ✅ T4-optimized deps installed")

In [ ]:
# CELL 4: Verify GPU + Models
import torch

print("=" * 55)
print("  GPU STATUS")
print("=" * 55)
print(f"  CUDA available: {torch.cuda.is_available()}")
print(f"  GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)")

print("\n" + "=" * 55)
print("  MODEL STATUS")
print("=" * 55)

for name, check in [
    ("Real-ESRGAN", lambda: __import__('realesrgan').RealESRGANer),
    ("GFPGAN", lambda: __import__('gfpgan').GFPGANer),
    ("face_recognition", lambda: __import__('face_recognition')),
    ("faster-whisper", lambda: __import__('faster_whisper')),
]:
    try:
        check()
        print(f"  ✅ {name}")
    except Exception:
        print(f"  ❌ {name}")

print("\n  Ready for AI pipeline!")

In [ ]:
# CELL 5: Apply Torchvision Compat + VRAM Cleanup Helper
import utils.torchvision_compat
print("  ✅ torchvision compat shim applied")

def clear_vram():
    """Free GPU memory between pipeline stages."""
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

# Create folders
for folder in ["input", "temp", "transcripts", "highlights", "shorts", "logs", "photos"]:
    Path(folder).mkdir(exist_ok=True)

print("  ✅ VRAM cleanup helper ready")

In [ ]:
# CELL 6: Start Worker + Tunnel
import time

subprocess.run("pkill -f 'python watcher.py' 2>/dev/null || true", shell=True)
subprocess.run("pkill -f 'lt --port' 2>/dev/null || true", shell=True)
time.sleep(1)

subprocess.Popen([sys.executable, "watcher.py"], stdout=open("watcher.log","w"), stderr=subprocess.STDOUT)
time.sleep(2)
subprocess.Popen(["lt", "--port", "5000"], stdout=open("tunnel.log","w"), stderr=subprocess.STDOUT)
time.sleep(5)

with open("tunnel.log") as f:
    for line in f:
        if "://" in line.strip():
            with open("kaggle_url.txt", "w") as out:
                out.write(line.strip())
            print(f"Tunnel URL: {line.strip()}")
            break

In [ ]:
# CELL 7: Monitor (keep this cell running)
print("=" * 55)
print("  WORKER IS ONLINE!")
print("=" * 55)
print("On Mac, run:")
print('  ./automate.sh "URL"')
print("  -> Option 2 (Remote Run)")

try:
    last_pos = 0
    last_inode = None
    while True:
        time.sleep(10)
        try:
            st = Path("watcher.log").stat()
        except FileNotFoundError:
            continue
        if last_inode is not None and st.st_ino != last_inode:
            last_pos = 0
        last_inode = st.st_ino
        with open("watcher.log", "r") as f:
            f.seek(last_pos)
            for l in f.readlines():
                if l.strip():
                    print(l.strip())
            last_pos = f.tell()
except KeyboardInterrupt:
    print("Stopped.")